# 🔢 Module 07: Embeddings & Vector Stores

---

## What Are Embeddings?

**Embeddings** convert text into numerical vectors (arrays of numbers) that capture **semantic meaning**. Similar texts have vectors that are close together in **vector space**.

```
"King"   → [0.23, -0.41, 0.87, ...]  (1536 numbers)
"Queen"  → [0.21, -0.39, 0.85, ...]  (very similar!)
"Car"    → [-0.83, 0.12, -0.44, ...] (very different)
```

### Why Do We Need Embeddings?

- LLMs can't search through documents by keyword efficiently
- Embeddings enable **semantic search**: "What is love?" matches "the feeling of deep affection"
- Foundation of **RAG** (Retrieval Augmented Generation)

---

## What Are Vector Stores?

**Vector Stores** are specialized databases that store embeddings and perform **similarity search**:

```
Query: "What is machine learning?"
   ↓ Embed
[0.12, -0.45, 0.78, ...]
   ↓ Search in vector store
Top 3 similar docs → pass to LLM for answer
```

---

## Popular Vector Stores

| Store | Type | Best For |
|-------|------|----------|
| **FAISS** | In-memory | Development, small datasets |
| **Chroma** | Local file | Development, prototyping |
| **Pinecone** | Cloud | Production, large scale |
| **Weaviate** | Cloud/self-hosted | Production |
| **Qdrant** | Cloud/self-hosted | Production |
| **pgvector** | PostgreSQL | Production + SQL |
| **OpenSearch** | Cloud/self-hosted | Enterprise |

---

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

%pip install -q faiss-cpu chromadb
print("Packages installed ✅")

## 1️⃣ Embedding Models

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# ============================================================
# OpenAI Embeddings — Most popular choice
# ============================================================
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",  # 1536 dimensions, cheap & fast
    # model_name="sentence-transformers/all-mpnet-base-v2" # 3072 dimensions, better quality
    # model="text-embedding-ada-002" # Legacy, 1536 dims
)

# Embed a single text
text = "The quick brown fox jumps over the lazy dog"
vector = embeddings.embed_query(text)

print(f"Text: '{text}'")
print(f"Vector dimensions: {len(vector)}")
print(f"First 5 values: {vector[:5]}")
print(f"Type: {type(vector[0]).__name__}")

In [ ]:
import numpy as np

# ============================================================
# Semantic Similarity — The power of embeddings!
# ============================================================

def cosine_similarity(v1, v2):
    """Compute cosine similarity between two vectors"""
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Texts to compare
texts = [
    "Machine learning is a type of artificial intelligence",  # Query
    "AI and ML are technologies that enable computers to learn",  # Similar
    "Deep learning uses neural networks with many layers",     # Somewhat related
    "The stock market was up 2% yesterday",                    # Unrelated
    "Python is a popular programming language for data science",# Tangentially related
]

# Embed all texts (batch)
vectors = embeddings.embed_documents(texts)

query_vector = vectors[0]
query_text = texts[0]

print(f"Query: '{query_text}'")
print("\nSimilarity to other texts:")
print("-" * 60)
for text, vector in zip(texts[1:], vectors[1:]):
    sim = cosine_similarity(query_vector, vector)
    bar = "█" * int(sim * 30)
    print(f"{sim:.3f} {bar}")
    print(f"       '{text[:60]}'")
    print()

In [ ]:
# ============================================================
# Google Embeddings (Alternative)
# ============================================================
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# 
# google_embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/embedding-001"
# )

# ============================================================
# HuggingFace Embeddings (Free, local!)
# ============================================================
# %pip install sentence-transformers
# from langchain_huggingface import HuggingFaceEmbeddings
#
# local_embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"  # Runs locally, FREE!
# )

print("See comments above for alternative embedding providers")

## 2️⃣ FAISS Vector Store

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# ============================================================
# Create documents
# ============================================================
docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications.", metadata={"topic": "langchain"}),
    Document(page_content="Python is a high-level, general-purpose programming language.", metadata={"topic": "python"}),
    Document(page_content="Neural networks are inspired by the structure of the human brain.", metadata={"topic": "ml"}),
    Document(page_content="Vector databases store high-dimensional vectors for similarity search.", metadata={"topic": "vectors"}),
    Document(page_content="RAG combines retrieval with language model generation.", metadata={"topic": "rag"}),
    Document(page_content="Transformers use attention mechanisms to process sequences.", metadata={"topic": "ml"}),
    Document(page_content="OpenAI provides GPT models through their API.", metadata={"topic": "llm"}),
    Document(page_content="Embeddings convert text into numerical vector representations.", metadata={"topic": "vectors"}),
]

# ============================================================
# Create FAISS index from documents (this calls the embedding API)
# ============================================================
vectorstore = FAISS.from_documents(docs, embeddings)

print(f"Vector store created with {vectorstore.index.ntotal} vectors")
print(f"Vector dimensions: {vectorstore.index.d}")

In [ ]:
# ============================================================
# Similarity Search
# ============================================================
query = "How do I convert text to numbers for AI?"

results = vectorstore.similarity_search(query, k=3)  # Top 3 results

print(f"Query: '{query}'")
print(f"\nTop {len(results)} results:")
print("-" * 50)
for i, doc in enumerate(results, 1):
    print(f"\n{i}. Topic: {doc.metadata['topic']}")
    print(f"   Content: {doc.page_content}")

In [ ]:
# ============================================================
# Similarity Search with Scores
# ============================================================
results_with_scores = vectorstore.similarity_search_with_score(query, k=4)

print(f"Query: '{query}'")
print("\nResults with similarity scores (lower = more similar for L2):")
print("-" * 60)
for doc, score in results_with_scores:
    print(f"Score: {score:.4f} | {doc.page_content[:60]}")

In [ ]:
# ============================================================
# Save and Load FAISS index (Persistence!)
# ============================================================
# Save to disk
vectorstore.save_local("faiss_index")
print("Index saved ✅")

# Load from disk (even after restart!)
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True  # Required flag
)

# Verify it works
result = loaded_vectorstore.similarity_search("What is RAG?", k=1)
print(f"Loaded index works! Result: '{result[0].page_content}'")

In [ ]:
# ============================================================
# Add documents to existing index
# ============================================================
new_docs = [
    Document(page_content="Agents can use tools to interact with the world.", metadata={"topic": "agents"}),
    Document(page_content="LangSmith helps debug and monitor LangChain applications.", metadata={"topic": "tooling"}),
]

vectorstore.add_documents(new_docs)
print(f"Updated index now has {vectorstore.index.ntotal} vectors")

## 3️⃣ Chroma Vector Store

In [ ]:
from langchain_chroma import Chroma

# ============================================================
# Chroma — Easy, persistent, supports metadata filtering
# ============================================================

# In-memory Chroma (deleted when script ends)
chroma_db = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="langchain_mastery"
)

# Persistent Chroma (survives restarts)
# chroma_db = Chroma.from_documents(
#     documents=docs,
#     embedding=embeddings,
#     persist_directory="./chroma_db"  # Saves to disk
# )

print(f"Chroma DB created with {chroma_db._collection.count()} documents")

In [ ]:
# ============================================================
# Metadata Filtering — Only search within a subset!
# ============================================================
# Search only in 'ml' topic documents
results = chroma_db.similarity_search(
    query="How do AI models learn?",
    k=3,
    filter={"topic": "ml"}  # Filter by metadata!
)

print("Search with metadata filter (topic=ml):")
for doc in results:
    print(f"  [{doc.metadata['topic']}] {doc.page_content}")

## 4️⃣ Vector Store as Retriever

In [ ]:
# ============================================================
# Convert vectorstore to retriever interface
# ============================================================
retriever = vectorstore.as_retriever(
    search_type="similarity",       # or "mmr" or "similarity_score_threshold"
    search_kwargs={
        "k": 3,                     # Return top 3
        # "score_threshold": 0.5    # Minimum similarity threshold
    }
)

# Use the retriever
docs_retrieved = retriever.invoke("What is an embedding?")

print("Retrieved documents:")
for doc in docs_retrieved:
    print(f"  - {doc.page_content}")

In [ ]:
# ============================================================
# MMR Search — Maximal Marginal Relevance
# Returns diverse results (avoids redundant similar chunks)
# ============================================================
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,      # Fetch 10 candidates
        "lambda_mult": 0.5  # Balance: 0=max diversity, 1=max relevance
    }
)

results = mmr_retriever.invoke("AI and machine learning")
print("MMR Results (diverse):")
for doc in results:
    print(f"  - [{doc.metadata['topic']}] {doc.page_content[:60]}")

## 5️⃣ Understanding Embedding Costs & Tradeoffs

| Model | Dimensions | Price per 1M tokens | Speed | Quality |
|-------|-----------|---------------------|-------|----------|
| text-embedding-3-small | 1536 | $0.02 | Fast | Good |
| text-embedding-3-large | 3072 | $0.13 | Fast | Best |
| text-embedding-ada-002 | 1536 | $0.10 | Fast | Good (legacy) |
| all-MiniLM-L6-v2 (HF) | 384 | FREE (local) | Very Fast | Decent |
| all-mpnet-base-v2 (HF) | 768 | FREE (local) | Fast | Good |

### 💡 Tips:
- **Development**: Use `text-embedding-3-small` or a free HuggingFace model
- **Production**: `text-embedding-3-small` is often good enough and cheap
- **High accuracy**: `text-embedding-3-large`
- **Privacy**: HuggingFace local models (no data leaves your machine)

In [ ]:
# Cleanup
import shutil
if os.path.exists("faiss_index"): shutil.rmtree("faiss_index")
print("Cleaned up ✅")

## ✅ Module 07 Summary

You've learned:
- ✅ What embeddings are and why they matter
- ✅ `HuggingFaceEmbeddings` (and alternatives)
- ✅ Cosine similarity for semantic search
- ✅ FAISS — fast in-memory vector store
- ✅ Chroma — persistent, filterable vector store
- ✅ Converting vectorstores to retrievers
- ✅ MMR for diverse results
- ✅ Embedding model tradeoffs

### 🚀 Next: [Module 08 — Retrievers & RAG](08_RAG_Retrieval_Augmented_Generation.ipynb)